# La Copa del Decano · Laboratorio causal de 120 minutos

**Basado en** `chapter-causalidad-master` del taller y sus tres capas. El simulador es sintético, la nota está en **0–50**, `pga_previo` en **0–5**, y `fatiga_latente` se mantiene oculta hasta la revelación docente.

**Archivos:** `sistema_academico_rector.csv`, `encuesta_estudiantes_juego.csv`, `cohorte_ab_test_2026.csv` descargados desde el juego (tu sala para los primeros dos). Ejecuta las celdas en **Google Colab** y carga los CSV en su panel de archivos.

**Ruta:** primero vemos una asociación → después un modelo muy predictivo → revisamos el DAG → finalmente un experimento aleatorizado. No interpretar SHAP como una intervención.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score
print("Entorno preparado. No se usarán datos personales reales.")


## Capa 1 · La trampa administrativa

El Rector observa el promedio previo H, la complejidad Z, las horas de bienestar B y la nota Y. Por sí solos no prueban qué ocurriría al imponer `do(B)`.


In [ ]:
rector = pd.read_csv("sistema_academico_rector.csv")
display(rector.head())
assert {"estudiante_id","examen_nro","complejidad","pga_previo","horas_bienestar_sem","nota_final"} <= set(rector.columns)
print("Exámenes:",len(rector),"· estudiantes:",rector.estudiante_id.nunique())
coef = np.polyfit(rector.horas_bienestar_sem,rector.nota_final,1)[0] if rector.horas_bienestar_sem.nunique()>1 else np.nan
print("Pendiente observacional de bienestar:",round(coef,3),"pt/h (NO causal)")
plt.scatter(rector.horas_bienestar_sem,rector.nota_final,alpha=.5)
plt.xlabel("Horas semanales bienestar B")
plt.ylabel("Nota Y (0–50)")
plt.show()


## Capa 2 · La trampa multivariable

La encuesta revela las horas de estudio X₁, el sueño S y el arquetipo conductual. **No controles automáticamente X₁ o S** si buscas el efecto total de B, porque pueden ser mediadores del camino causal. El arquetipo es una decisión de interfaz y no reemplaza una variable causal medida.


In [ ]:
encuesta = pd.read_csv("encuesta_estudiantes_juego.csv")
d = rector.merge(encuesta,on=["estudiante_id","examen_nro"],validate="one_to_one")
display(d.head())
print("Filas vinculadas:",len(d),"— debes usar ambos identificadores para evitar cruces duplicados.")


In [ ]:
covariables = ["pga_previo","complejidad","horas_bienestar_sem","horas_estudio","horas_sueno"]
X=d[covariables]; y=d.nota_final
if d.estudiante_id.nunique()>=4:
    train,test=next(GroupShuffleSplit(n_splits=1,test_size=.3,random_state=2026).split(X,y,groups=d.estudiante_id))
    model=RandomForestRegressor(n_estimators=60,min_samples_leaf=2,random_state=2026,n_jobs=-1)
    model.fit(X.iloc[train],y.iloc[train])
    pred=model.predict(X.iloc[test])
    print("MAE",round(mean_absolute_error(y.iloc[test],pred),3),"R²",round(r2_score(y.iloc[test],pred),3))
    print("Importancia predictiva (NO causal):",dict(zip(covariables,model.feature_importances_.round(3))))
else:
    print("Muy pocos estudiantes distintos para una validación train/test por persona; evita confundir autocorrelación con generalización.")


**Discusión guiada y cerrada:** ¿controlar estudio y sueño garantiza el efecto TOTAL de bienestar? **No.** El DAG incluye H→B, H→Y, B→X₁, B→S, B→U, X₁→S, X₁→U, S→U, U→Y y B→Y. Los mediadores posteriores pueden producir *bad control*. SHAP, aunque explique predicciones, tampoco identifica `do(B)`.


## Experimento A/B · Política del Rector

Se sortea **T a nivel de estudiante**, no a nivel de examen. T=1 impone 4 horas semanales de bienestar y T=0 impone 0. Los 15 exámenes comparten asignación, por tanto los errores deben calcularse **por estudiante** para evitar pseudorreplicación.


In [ ]:
ab = pd.read_csv("cohorte_ab_test_2026.csv")
assert {"estudiante_id","examen_nro","tratamiento","nota_final","horas_bienestar_sem"} <= set(ab.columns)
per_student=ab.groupby(["estudiante_id","tratamiento"],as_index=False).nota_final.mean()
treated=per_student.loc[per_student.tratamiento==1,"nota_final"]
control=per_student.loc[per_student.tratamiento==0,"nota_final"]
assert len(treated)>1 and len(control)>1, "Faltan estudiantes por brazo."
diff=treated.mean()-control.mean()
se=np.sqrt(treated.var(ddof=1)/len(treated)+control.var(ddof=1)/len(control))
print(f"Tratados: {len(treated)} · Controles: {len(control)}")
print(f"Efecto observado: {diff:.3f} puntos")
print(f"IC95 aproximado por estudiante: [{diff-1.96*se:.3f},{diff+1.96*se:.3f}]")
print("No es un efecto individual observado; el ITE verdadero solo está en el archivo docente sintético.")


In [ ]:
balance=ab.groupby("tratamiento").pga_previo.mean()
print("PGA previo por grupo T=0/1:",balance.to_dict())
print("La aleatorización mejora comparabilidad en promedio, pero una muestra finita puede estar desbalanceada.")
print("Pregunta cerrada: ¿el signo del efecto observacional de la Capa 1 prueba el ATE? NO.")


## Conexión con DML y EconML

El **DML** con *cross-fitting* permite ajustar modelos auxiliares para variables anteriores al tratamiento y usar un *score* ortogonal, **pero no corrige confusión no observada automáticamente**. Un CATE estimado tampoco es el contrafactual individual verdadero.

**EconML del juego V2 anterior** corresponde a otro DGP. No debe compararse directamente con las notas de la Copa. Se debe reentrenar y evaluar sobre datos diseñados para esta cohorte antes de afirmar un benchmark propio.


## Lista de verificación

- ¿Distingues la asociación del Rector del efecto de la intervención?
- ¿Puedes señalar qué variables del DAG son anteriores y cuáles posteriores?
- ¿La asignación A/B fue a nivel de estudiante?
- ¿El intervalo usa estudiantes independientes en lugar de los 450 exámenes como si fueran 450 personas?
- ¿Las recomendaciones comunican incertidumbre, capacidad y límites?

Para el docente existe un CSV adicional de resultados potenciales y fatiga sintética. **No se proporciona a los participantes antes del cierre conceptual.**
